In [14]:
import numpy as np
from numpy.linalg import inv
from sklearn.preprocessing import PolynomialFeatures

In [15]:
X = np.array([[3,3.5], 
              [9,2.3],
              [7,5],
              [2,7]])
y = np.array([[-1],
              [1],
              [-1],
              [-1]])

### Number of polynomial terms
Let **p** be the number of raw features (`X.shape[1]`) and **n** be the degree (`order`). The number of polynomial terms, including the intercept and all powers/interactions up to degree n, is:

$$
\boxed{d = \binom{p+n}{n} = \frac{(p+n)!}{p!\,n!}}
$$

For the current **2 features** and **order 2**:

$$
d = \binom{2+2}{2} = \frac{4!}{2!\,2!} = 6.
$$

The terms are:

$$
1,\quad x_1,\quad x_2,\quad x_1^2,\quad x_1x_2,\quad x_2^2.
$$

`PolynomialFeatures` includes the intercept by default. With `include_bias=False`, subtract **1**. After expansion, `X_poly.shape[1]` gives the actual number of terms and coefficients per output/class.


In [ ]:
order = 2
poly = PolynomialFeatures(degree=order)
X_poly = poly.fit_transform(X)

In [17]:
# Keep targets as columns; accepts a 1-D target vector too.
y = np.array(y, dtype=float)
if len(y.shape) == 1:
    y = y.reshape(-1, 1)
assert len(y.shape) == 2 and y.shape[0] == X_poly.shape[0], "X and y must have the same number of rows."

# MANUAL IF/ELSE: select the inverse using the polynomial matrix shape.
m, d = X_poly.shape
if m >= d:
    # Tall or square matrix: LEFT inverse (full column rank required).
    assert np.linalg.matrix_rank(X_poly) == d, "Left inverse does not exist: columns are dependent."
    w = inv(X_poly.T @ X_poly) @ X_poly.T @ y
    print("Using LEFT inverse")
else:
    # Wide matrix: RIGHT inverse (full row rank required).
    assert np.linalg.matrix_rank(X_poly) == m, "Right inverse does not exist: rows are dependent."
    w = X_poly.T @ inv(X_poly @ X_poly.T) @ y
    print("Using RIGHT inverse")

# Display every polynomial term with its weight.
raw_names = [f"x{i + 1}" for i in range(X.shape[1])]
terms = poly.get_feature_names_out(raw_names)
print("Polynomial matrix shape:", X_poly.shape)
for output in range(w.shape[1]):
    if w.shape[1] > 1:
        print(f"\nOutput {output + 1}:")
    for i in range(len(terms)):
        print(f"{terms[i]}: {w[i, output]:.6f}")


Using RIGHT inverse
Polynomial matrix shape: (4, 6)
1: -0.078577
x1: -0.159948
x2: -0.168614
x1^2: 0.048310
x1 x2: -0.054657
x2^2: 0.023484


Prediction; edit the X_test with features you would like to test with

In [18]:
X_test = np.array([[10, 4.5]])
X_test_poly = poly.transform(X_test)
y_pred = X_test_poly @ w

print(f"Predicted y: {y_pred[0, 0]:.4f}")

Predicted y: 0.4101
